In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ..

assert os.getcwd().endswith(root_dir)

In [ ]:
nvae_m2m_extended = "logs/nvae_corrector_acdc/default-extended/checkpoints/epoch=20-step=17976.ckpt"
cnvae_i2m = "logs/cnvae_acdc/best/checkpoints/epoch=83-step=17976.ckpt"
unet = "logs/unet_acdc/default-unet/checkpoints/epoch=52-step=2862.ckpt"

In [ ]:
from arch.nvae_corrector.nvae_corrector import NVAECorrector
from arch.cnvae.cnvae import CNVAE
from arch.unet.unet import UNet

import lightning as L

from utils.const import SEED
from data_modules.acdc import ACDC3DWithPredictedMaskDataModule
from utils.utils import setup_device

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDC3DWithPredictedMaskDataModule()

# Load models
models_nvae_m2m_extended = NVAECorrector.load_from_checkpoint(nvae_m2m_extended)
models_cnvae_i2m = CNVAE.load_from_checkpoint(cnvae_i2m)
models_unet = UNet.load_from_checkpoint(unet)

# Reseed after preprocessing data
L.seed_everything(SEED)

In [ ]:
loader_test = data_module.test_dataloader()
y, y_pred, _, _ = next(iter(loader_test))
print(y.shape, y_pred.shape)

In [ ]:
# Do not use a data loader, since scans are encapsulated. Furthermore, not
# further preprocessing is done in Dataset.__getitem__.

data = data_module.data_test

print(data.scans[0].shape)
print(data.masks[0].shape)
print(data.masks_pred[0].shape)

print(len(data))

In [ ]:
import torch

from utils.utils import discretise

y_hat_nvae_m2m_extendeds = []
y_hat_cnvae_i2ms = []
y_hat_unets = []


with torch.no_grad():
    models = [models_nvae_m2m_extended, models_cnvae_i2m, models_unet]
    
    for model in models:
        model.eval()
        model.to(device)
        
    i = 1
    
    xs = data.scans[i]
    ys = data.masks[i]
    y_preds = data.masks_pred[i]
    
    for x, y, y_pred in zip(xs, ys, y_preds):
        # Individual 2D slice
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)
        y_pred = y_pred.unsqueeze(0)

        x = x.to(device)
        y = y.to(device)
        y_pred = y_pred.to(device)
        
        y_hat_logits_nvae_m2m_extended, _, _, _, _ = models_nvae_m2m_extended(y_pred, test=True)
        y_hat_onehot_nvae_m2m_extended = discretise(y_hat_logits_nvae_m2m_extended)
        
        y_hat_logits_cnvae_i2m = models_cnvae_i2m.inference(x)
        y_hat_onehot_cnvae_i2m = discretise(y_hat_logits_cnvae_i2m)
        
        y_hat_logits_unet = models_unet(x)
        y_hat_onehot_unet = discretise(y_hat_logits_unet)
        
        y_hat_nvae_m2m_extendeds.append(y_hat_onehot_nvae_m2m_extended)
        y_hat_cnvae_i2ms.append(y_hat_onehot_cnvae_i2m)
        y_hat_unets.append(y_hat_onehot_unet)

In [ ]:
print(xs.shape)
print(ys.shape)
print(y_hat_nvae_m2m_extendeds[0].shape)
print(y_hat_cnvae_i2ms[0].shape)
print(y_hat_unets[0].shape)

In [ ]:
index = 5

In [ ]:
torch.argmax(y_hat_nvae_m2m_extendeds[index], dim=1).unsqueeze(1)

In [ ]:
# Choose the sample to view

from utils.utils import show_samples, show_scans

figsize = (6, 6)
# figsize = (1, 1)

show_scans(
    xs[index],
    ncol=1,
    figsize=figsize,
)

show_samples(
    torch.argmax(ys[index].unsqueeze(0), dim=1).unsqueeze(1),
    rgb=False,
    ncol=1,
    figsize=figsize,
)

show_samples(
    torch.argmax(y_hat_nvae_m2m_extendeds[index], dim=1).unsqueeze(1),
    rgb=False,
    ncol=1,
    figsize=figsize,
)

show_samples(
    torch.argmax(y_hat_cnvae_i2ms[index], dim=1).unsqueeze(1),
    rgb=False,
    ncol=1,
    figsize=figsize,
)

show_samples(
    torch.argmax(y_hat_unets[index], dim=1).unsqueeze(1),
    rgb=False,
    ncol=1,
    figsize=figsize,
)